In [ ]:
# Install the pdfplumber module
!pip install pdfplumber -q

In [ ]:
# import the necessary libraries
import sys
import pdfplumber
import pandas as pd
import re
import os
from google.colab import drive

try:
    # 1. Mount the drive
    drive.mount('/content/drive')

    # 2. Set your path variable (Corrected name to Pieter Linden)
    path = "/content/drive/MyDrive/2.) Beacon Of Hope/1.) Stakeholders/6.) Pieter Linden/1.) Costco Invoice Challenge"

    # 3. Change the current working directory to your project folder
    os.chdir(path)

    # 4. Verify the location and list the files
    print(f"Current Directory: {os.getcwd()}")
except:
    print(f"❌ FILE NOT FOUND: {path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current Directory: /content/drive/MyDrive/2.) Beacon Of Hope/1.) Stakeholders/6.) Pieter Linden/1.) Costco Invoice Challenge


In [ ]:
def clean_costco_final_touch(path):
    all_data = []

    # Regex patterns
    date_pattern = r'\b(0[1-9]|1[0-2])-(0[1-9]|[12][0-9]|3[01])\b'
    item_id_pattern = r'\b\d{5,}\b' # Products usually start with a 5-7 digit ID

    # Maps for Data Normalization
    tax_map = {'3': 'Grocery Tax', 'Y': 'Standard Tax', 'None': 'Discount'}
    dept_map = {
        '12': 'Candy & Snacks', '13': 'Dry Grocery', '14': 'Beverages',
        '19': 'Deli & Prepared Foods', '23': 'Home & Hardlines',
        '25': 'Automotive & Tires', '61': 'Fresh Meat'
    }

    current_date = None
    current_year = None

    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if not text: continue

            lines = text.split('\n')
            for line in lines:
                # 1. Update Current Date
                date_match = re.search(date_pattern, line)
                if date_match:
                    raw_date = date_match.group(0)
                    month = int(raw_date.split('-')[0])
                    current_year = "2021" if month >= 10 else "2022"
                    current_date = f"{current_year}-{raw_date}"

                # 2. Identify Valid Product Rows
                amount_match = re.findall(r'-?\d+\.\d{2}', line)

                if amount_match and current_date:
                    # We only accept the row if it has an Item ID OR if it's a negative amount (Discount)
                    has_item_id = re.search(item_id_pattern, line)
                    amount = float(amount_match[-1])

                    if has_item_id or amount < 0:
                        # Extract Dept Code (2-digit number)
                        dept_codes = re.findall(r'\b\d{2}\b', line)
                        clean_dept_codes = [c for c in dept_codes if c not in current_date]
                        dept_code = clean_dept_codes[0] if clean_dept_codes else "N/A"

                        # CLEAN PRODUCT NAME
                        # Step A: Remove the amount and date fragments
                        p_name = line.replace(amount_match[-1], '')
                        if date_match: p_name = p_name.replace(date_match.group(0), '')

                        # Step B: Remove leading/trailing numeric IDs and technical codes
                        p_name = re.sub(r'\b\d{4,}\b', '', p_name) # Remove IDs (4+ digits)
                        p_name = re.sub(r'\b\d{2}\b', '', p_name)   # Remove Dept codes

                        # Step C: Final Polish
                        p_name = p_name.replace('2021-', '').replace('2022-', '')
                        p_name = p_name.replace('/', '').strip()

                        # Capture Tax Status
                        tax_char = 'None'
                        if line.strip().endswith(' 3'): tax_char = '3'
                        elif line.strip().endswith(' Y'): tax_char = 'Y'

                        all_data.append({
                            'PurchaseDate': current_date,
                            'Year': int(current_year),
                            'ProductName': p_name[:30].strip(),
                            'Amount': amount,
                            'IsDiscount': amount < 0,
                            'Adjustment': tax_map.get(tax_char, 'Non-Taxable' if amount >=0 else 'Discount'),
                            'Department': dept_map.get(dept_code, f"Dept {dept_code} (Other)")
                        })

    # Append all of the data
    df = pd.DataFrame(all_data)

    # Strip away the first leading characters of the product names
    lambda_function = lambda x: re.sub(r'^[^A-Za-z]*', '', x)
    df["ProductName"] = df["ProductName"].apply(lambda_function)

    return df.reset_index(drop=True)

In [ ]:
# Calling the cleaning function
try:
    pdf_path = "Costco Invoice.pdf"
    output_csv = pdf_path.replace(".pdf", ".csv")

    print("🚀 Starting the secured data cleaning...")
    df_cleaned = clean_costco_final_touch(pdf_path)

    if not df_cleaned.empty:
        df_cleaned.to_csv(output_csv, index=False)
        print(f"\n✅ SUCCESS! Produced {len(df_cleaned)} rows for your dashboard.")
        print("\nPreview of the secured output:")
        print(df_cleaned.head(10))
    else:
        print("\n❌ ERROR: No data rows found. Check the PDF formatting.")
except Exception as e:
    print(f"❌ AN ERROR OCCURRED: {e}")

🚀 Starting the secured data cleaning...

✅ SUCCESS! Produced 63 rows for your dashboard.

Preview of the secured output:
  PurchaseDate  Year                 ProductName  Amount  IsDiscount  \
0   2021-10-09  2021      KS ORG GRD  4  142   3  151.60       False   
1   2021-10-09  2021         KS ORG  4  142 3  3   22.47       False   
2   2021-10-09  2021     KS ORGANIC  4  142 1  3    9.49       False   
3   2021-10-09  2021       DURACELL AA  4 142 -2   -6.00        True   
4   2021-10-09  2021    DURACELL AA  4  142 2  Y   33.98       False   
5   2021-10-09  2021  LASKO SOLAIRE  4  142 1  Y   59.99       False   
6   2021-10-09  2021     KS ORG EVOO  202   8  3   95.92       False   
7   2021-10-16  2021       DURACELL AAA  202  -1   -3.00        True   
8   2021-10-16  2021    DURACELL AAA  202   1  Y   16.99       False   
9   2021-10-16  2021       ORG PITED  9  120   3  100.62       False   

     Adjustment        Department  
0   Grocery Tax       Dry Grocery  
1   Grocery Ta